In [3]:
# 8_label_clusters.ipynb
#
# For each cluster produced by 6_cluster.ipynb, calls the OpenAI Chat API
# to generate a short title and 2-3 sentence description per cluster,
# then saves enriched results to data/6_cluster/<name>_described.csv.
#
# Uses group-level baselines and categorical distributions from step 7
# so the LLM can describe what is *distinctive* about each cluster
# compared to the broader employment group in that LA.
#
# All clusters for a single LA are sent in ONE API call.
#
# Requires OPENAI_API_KEY in environment or .env file.

import sys, os, time, json, shutil
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
from pathlib import Path
from tqdm import tqdm

import importlib
import data_pipeline.config_paths as _cp
importlib.reload(_cp)
from data_pipeline.config_paths     import DATA_FOLDER, USE_FOUR_LA_SUBSET, FOUR_LA_CODES
import data_pipeline.config_variables as _cv
import data_pipeline.config_cluster   as _cc
importlib.reload(_cv)
importlib.reload(_cc)

from data_pipeline.config_variables import VARIABLE_MAP
from data_pipeline.config_cluster   import WAVE, CLUSTER_VARS
from openai import OpenAI

# ── Load .env if present ──────────────────────────────────────────────────────
try:
    from dotenv import load_dotenv
    load_dotenv(Path('..') / '.env', override=False)
    print("Loaded .env")
except ImportError:
    pass

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "")
if not OPENAI_API_KEY:
    raise EnvironmentError(
        "OPENAI_API_KEY not set. Export it in your shell or add it to .env:\n"
        "  export OPENAI_API_KEY=sk-..."
    )

# ── Config ────────────────────────────────────────────────────────────────────
MODEL          = "gpt-4.1-mini"   # 32 768 output-token limit; cheap & fast
MAX_OUT_TOKENS = 32_000           # hard cap, safely under the model limit
CLUSTER_CSV = Path(f"../{DATA_FOLDER}/6_cluster/LA_london_clusters.csv")
OUTPUT_CSV  = CLUSTER_CSV.parent / (CLUSTER_CSV.stem + "_described.csv")
BASELINES_CSV     = Path(f"../{DATA_FOLDER}/7_group_averages/group_baselines.csv")
DISTRIBUTIONS_CSV = Path(f"../{DATA_FOLDER}/7_group_averages/group_distributions.csv")
MAX_RETRIES = 3
RETRY_DELAY = 5

# Same geography scope as 6_cluster / 7_group_averages (config_paths)
UNIT_FILTER = list(FOUR_LA_CODES) if USE_FOUR_LA_SUBSET else "london"

client = OpenAI(api_key=OPENAI_API_KEY)
print(f"Model:       {MODEL}")
print(f"Cluster CSV: {CLUSTER_CSV}")
print(f"Output CSV:  {OUTPUT_CSV}")

# ── Load CSVs ─────────────────────────────────────────────────────────────────
df = pd.read_csv(CLUSTER_CSV)
if UNIT_FILTER is None:
    pass
elif UNIT_FILTER == "london":
    df = df[df["unit_id"].astype(str).str.startswith("E09")]
elif isinstance(UNIT_FILTER, list):
    df = df[df["unit_id"].isin(set(UNIT_FILTER))]
else:
    raise ValueError("UNIT_FILTER must be None, 'london', or a list of LA codes")
if df.empty:
    raise ValueError(
        "No cluster rows after UNIT_FILTER — check CLUSTER_CSV matches config_paths "
        "(re-run 6_cluster with the same USE_FOUR_LA_SUBSET)."
    )
META_COLS = {'tribe_label', 'size', 'unit_id', 'la_name', 'cluster_level', 'group'}
print(f"Loaded {len(df)} cluster rows across {df['unit_id'].nunique()} units [USE_FOUR_LA_SUBSET={USE_FOUR_LA_SUBSET}]")

# Group baselines (from step 7)
df_baselines = pd.read_csv(BASELINES_CSV) if BASELINES_CSV.exists() else pd.DataFrame()
df_dist      = pd.read_csv(DISTRIBUTIONS_CSV) if DISTRIBUTIONS_CSV.exists() else pd.DataFrame()
_uids = set(df["unit_id"].unique())
if not df_baselines.empty:
    df_baselines = df_baselines[df_baselines["unit_id"].isin(_uids)]
if not df_dist.empty:
    df_dist = df_dist[df_dist["unit_id"].isin(_uids)]
if not df_baselines.empty:
    print(f"Loaded {len(df_baselines)} group baselines from step 7")
if not df_dist.empty:
    print(f"Loaded {len(df_dist)} distribution rows from step 7")

# ── Prompts (clustering vars from config_cluster; labels from VARIABLE_MAP) ───
_CLUSTERING_LABELS = ", ".join(VARIABLE_MAP[c] for c in CLUSTER_VARS)
_PUBLIC_SERVICE_LABELS = ", ".join(
    VARIABLE_MAP[c] for c in ("locserc", "locserd", "locsere") if c in VARIABLE_MAP
)
_DIGITAL_KEYS = ("netpusenew", "netuse", "onlinebank", "onlinebuy", "smartmob", "laptop")
_DIGITAL_LABELS = ", ".join(VARIABLE_MAP[c] for c in _DIGITAL_KEYS if c in VARIABLE_MAP)

SYSTEM_PROMPT = f"""You are a social researcher specialising in UK population demographics.
You will be given the statistical profiles of several population clusters from a single
Local Authority — spanning multiple employment groups (Employed, Retired, Student, etc.) —
derived from the UK Household Longitudinal Study (UKHLS).

For each employment group you will first see a GROUP BASELINE showing the averages and
categorical breakdowns across ALL people in that group within the LA. Then each cluster's
stats follow. Your job is to identify what makes each cluster distinctive compared to
its group baseline and the other clusters.

**Clustering variables** (k-means used ONLY these):
{_CLUSTERING_LABELS}

**Title rules**
- Exactly 3–5 words, vivid and memorable.
- The title must ONLY allude to the clustering variables above. Do NOT mention employment,
  qualification, health, gender, commute, neighbourhood cohesion, public services,
  digital behaviour, or any other non-clustering field in the title.

**Description structure**
Each "description" must be a single string with exactly three sections, in this order,
each separated from the next by a blank line.

Each section MUST begin with its heading line copied EXACTLY from the numbered list below — same
words, same punctuation, same **bold** placement. Do NOT use older or shorter headings such as
"**Public services:**", "**Digital:**", or "**What can we say about this group's use of public services:**".

1) **What defines this cluster:**
2) What can we say about this group's use of public **Services**:
3) What can we say about their **Digital life**:

Under heading 1: Answer using the clustering variables only ({_CLUSTERING_LABELS}). Say what distinguishes
this cluster from the group baseline and from other clusters — do not bring in other fields here.

Under heading 2: Draw on satisfaction with local public transport, shopping, and leisure facilities relative to
the group baseline and other clusters. Use these fields: {_PUBLIC_SERVICE_LABELS}.
Ratings are typically 1–5; higher = better satisfaction unless a field label implies otherwise.

Under heading 3: Cover internet use, online banking/shopping, smartphone and laptop access relative to the baseline
where relevant. Use these fields: {_DIGITAL_LABELS}.

Optional context for other stats in the input (do not base the title on these):
- "Highest qualification": 1=Degree … 6=No qualification. Lower = higher qualification.
- "Job type (NS-SEC 8)": 1=Higher managerial … 8=Routine. Lower = higher class.
- "Mental/Physical health score (SF-12)": 0–100; higher = better health.
- Neighbourhood Cohesion Index 1–5; higher = stronger community.

Respond with a JSON object with a single key "clusters" whose value is an array.
Each array element corresponds to one input cluster and must contain exactly:
  "tribe_label"  — copied verbatim from the input (used to match results back)
  "title"        — per the title rules above
  "description"  — per the three-section structure above

Respond with valid JSON only — no markdown code fences, no explanation outside the JSON.
Escape any line breaks inside JSON string values as \n."""

# Sanity check when you run this cell — these lines must match the model instructions above.
for _ln in SYSTEM_PROMPT.splitlines():
    _s = _ln.strip()
    if _s.startswith("**What defines") or "public **Services**" in _ln or "**Digital life**" in _ln:
        print("  [prompt heading]", _ln)


def _format_val(val):
    """Format a value for prompt display."""
    if pd.isna(val):
        return None
    if isinstance(val, float) and val == int(val):
        return str(int(val))
    if isinstance(val, float):
        return f"{val:.1f}"
    return str(val)


def _build_baseline_block(unit_id: str, group: str) -> list[str]:
    """Build the GROUP BASELINE text block for one (unit_id, group)."""
    lines = []
    if df_baselines.empty:
        return lines

    bl = df_baselines[(df_baselines['unit_id'] == unit_id) & (df_baselines['group'] == group)]
    if bl.empty:
        return lines

    bl_row = bl.iloc[0]
    lines.append(f"  === GROUP BASELINE: all {group} in this LA (n={int(bl_row['size']):,}) ===")

    # Continuous/binary stats from the baseline row
    skip = META_COLS | {'tribe_label', 'size', 'unit_id', 'la_name', 'cluster_level', 'group'}
    for col in bl_row.index:
        if col in skip:
            continue
        val = _format_val(bl_row[col])
        if val is not None:
            lines.append(f"    {col}: {val}")

    # Full categorical distributions
    if not df_dist.empty:
        dist = df_dist[(df_dist['unit_id'] == unit_id) & (df_dist['group'] == group)]
        if not dist.empty:
            for var_name, var_dist in dist.groupby('variable', sort=False):
                var_dist = var_dist.sort_values('pct', ascending=False)
                parts = [f"{r['category']}: {r['pct']}%" for _, r in var_dist.iterrows() if r['pct'] >= 1.0]
                if parts:
                    lines.append(f"    {var_name} breakdown: {' | '.join(parts)}")

    lines.append("")
    return lines


def build_la_prompt(unit_id: str, la_name: str, rows: pd.DataFrame) -> str:
    """Build a single prompt containing baselines + all clusters for one LA."""
    lines = [
        f"Local Authority: {la_name} ({unit_id})",
        f"Clusters to label: {len(rows)}",
        "",
    ]

    # Insert group baselines before the cluster details
    seen_groups = set()
    for _, row in rows.iterrows():
        group = row.get('group', 'Unknown')
        if group not in seen_groups:
            seen_groups.add(group)
            lines.extend(_build_baseline_block(unit_id, group))

    for _, row in rows.iterrows():
        lines += [
            f"--- tribe_label: {row['tribe_label']} ---",
            f"  Employment group: {row.get('group', 'Unknown')}",
            f"  Population size:  {int(row['size']):,}",
        ]
        for col in [c for c in row.index if c not in META_COLS]:
            val = _format_val(row[col])
            if val is not None:
                lines.append(f"  {col}: {val}")
        lines.append("")
    return "\n".join(lines)


# ── One API call per LA ───────────────────────────────────────────────────────
# (unit_id, tribe_label) -> {"gpt_title": ..., "gpt_description": ...}
label_map: dict[tuple, dict] = {}

la_name_col = "la_name" if "la_name" in df.columns else "unit_id"
unit_ids = df["unit_id"].unique().tolist()

_call_num = 0
for unit_id in tqdm(unit_ids, desc="Labelling LAs"):
    la_rows = df[df["unit_id"] == unit_id]
    la_name = la_rows[la_name_col].iloc[0]
    prompt  = build_la_prompt(unit_id, la_name, la_rows)

    _call_num += 1
    if _call_num == 1 or _call_num % 50 == 0:
        print(f"\n{'='*60}\n[Call #{_call_num}] Prompt for {la_name} ({unit_id}):\n{'='*60}")
        print(prompt)
        print('='*60)

    response_text = None
    for attempt in range(MAX_RETRIES):
        try:
            resp = client.chat.completions.create(
                model=MODEL,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user",   "content": prompt},
                ],
                response_format={"type": "json_object"},
                temperature=0.7,
                max_tokens=min(450 * len(la_rows) + 400, MAX_OUT_TOKENS),
            )
            response_text = resp.choices[0].message.content
            break
        except Exception as e:
            wait = RETRY_DELAY * (2 ** attempt)
            print(f"\n  Error on {unit_id}: {e}  — retrying in {wait}s")
            time.sleep(wait)

    if not response_text:
        continue

    try:
        parsed = json.loads(response_text)
        # Expect {"clusters": [...]} but tolerate a bare list or other wrapper key
        items = parsed.get("clusters") or next(
            (v for v in parsed.values() if isinstance(v, list)), []
        )
        for item in items:
            key = (unit_id, item.get("tribe_label", ""))
            label_map[key] = {
                "gpt_title":       (item.get("title", "") or "").strip() or None,
                "gpt_description": (item.get("description", "") or "").strip() or None,
            }
    except (json.JSONDecodeError, AttributeError) as exc:
        print(f"\n  Parse error for {unit_id}: {exc}")

successful = sum(1 for v in label_map.values() if v["gpt_title"])
print(f"\nCompleted {len(unit_ids)} LA calls — {successful}/{len(df)} clusters labelled")

# ── Merge back and save ───────────────────────────────────────────────────────
df["gpt_title"]       = df.apply(lambda r: label_map.get((r["unit_id"], r["tribe_label"]), {}).get("gpt_title"),       axis=1)
df["gpt_description"] = df.apply(lambda r: label_map.get((r["unit_id"], r["tribe_label"]), {}).get("gpt_description"), axis=1)

df.to_csv(OUTPUT_CSV, index=False)
print(f"Saved {len(df)} rows to {OUTPUT_CSV}")
display(df[["unit_id", "group", "tribe_label", "size", "gpt_title", "gpt_description"]].head(12))

# ── Copy to api/ for deployment ───────────────────────────────────────────────
API_CLUSTERS_DIR = Path('..') / 'api' / 'data' / 'clusters'
API_CLUSTERS_DIR.mkdir(parents=True, exist_ok=True)
shutil.copy(OUTPUT_CSV, API_CLUSTERS_DIR / OUTPUT_CSV.name)
print(f'Copied {OUTPUT_CSV.name}  ->  api/data/clusters/')



🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  
  FOUR-LA SUBSET ACTIVE — Newham, Tower Hamlets, Islington, Hounslow only (4 LAs)
  Set USE_FOUR_LA_SUBSET = False for all London or full UK.
🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  

Loaded .env
Model:       gpt-4.1-mini
Cluster CSV: ../data/6_cluster/LA_london_clusters.csv
Output CSV:  ../data/6_cluster/LA_london_clusters_described.csv
Loaded 55 cluster rows across 4 units [USE_FOUR_LA_SUBSET=True]
Loaded 24 group baselines from step 7
Loaded 2336 distribution rows from step 7
  [prompt heading] 2) What can we say about this group's use of public **Services**:
  [prompt heading] 3) What can we say about their **Digital life**:


Labelling LAs:   0%|          | 0/4 [00:00<?, ?it/s]


[Call #1] Prompt for Hounslow (E09000018):
Local Authority: Hounslow (E09000018)
Clusters to label: 13

  === GROUP BASELINE: all Employed in this LA (n=84,192) ===
    Age in years: 47.5
    Gender: Male
    Gender %: 56
    Ethnic group: White
    Ethnic group %: 58
    Highest qualification: 2.5
    Total monthly personal income (gross): 2861.5
    Job type (NS-SEC 8): 3.6
    Number of own children in household: 0.7
    Has children: 0.4
    Immigrant generation: Not stated
    Immigrant generation %: 99
    Household size: 3.4
    Monthly net pay (take-home): 1456.7
    Personal income — administrative / derived (UKHLS): 2099.3
    Highest qualification — administrative / derived (UKHLS): 2.7
    Mental health score (SF-12 MCS): 45.3
    Physical health score (SF-12 PCS): 50.5
    Buckner Neighbourhood Cohesion Index: 3.4
    Standard of local services: Public transport: 2.7
    Standard of local services: Shopping: 2.2
    Standard of local services: Leisure: 2.5
    Internet us

Labelling LAs: 100%|██████████| 4/4 [14:23<00:00, 215.76s/it]


Completed 4 LA calls — 55/55 clusters labelled
Saved 55 rows to ../data/6_cluster/LA_london_clusters_described.csv


,unit_id,group,tribe_label,size,gpt_title,gpt_description
0,E09000018,Employed,Employed 1,17052,Mid-Age White Families,1) **What defines this cluster:** This cluster...
1,E09000018,Employed,Employed 2,23927,Older White Small Households,1) **What defines this cluster:** This group i...
2,E09000018,Employed,Employed 3,7858,Mid-Age Other Asian Workers,1) **What defines this cluster:** This cluster...
3,E09000018,Employed,Employed 4,16445,Younger White Singles,1) **What defines this cluster:** This cluster...
4,E09000018,Employed,Employed 5,11091,High-Income White Males,1) **What defines this cluster:** This group i...
5,E09000018,Employed,Employed 6,7819,Pakistani-Bangladeshi Large Households,1) **What defines this cluster:** Younger (36....
6,E09000018,Retired,Retired 1,18839,Oldest White Singles,1) **What defines this cluster:** This cluster...
7,E09000018,Retired,Retired 2,4064,Older Indian Large Households,1) **What defines this cluster:** This group i...
8,E09000018,Unemployed,Unemployed 1,4311,Older White Small Households,1) **What defines this cluster:** Older (52.8 ...
9,E09000018,Unemployed,Unemployed 2,3727,Younger Pakistani-Bangladeshi Families,1) **What defines this cluster:** Younger (39....


Copied LA_london_clusters_described.csv  ->  api/data/clusters/
